In [167]:
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from urllib.parse import urljoin
import re
from translate import Translator
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 100)  # Maximum number of columns to display
pd.set_option('display.max_colwidth', 200)  # Maximum width of column values

In [168]:
def convert_image(image_path, image_format):
    image = Image.open(image_path)
    image = image.convert('RGB')
    image.save(image_path, image_format)
    image.close()

def download_image(url, save_path, image_format):
    response = requests.get(url)
    if response.status_code == 200:
        with open(save_path, 'wb') as file:
            file.write(response.content)
        # end
        
        if url.split('.')[-1] != 'jpg':
            convert_image(save_path, image_format)
        # end
        
        #print(f"Image downloaded and converted to {image_format} successfully: {save_path}.")
    else:
        1
        #print("Failed to download the image.")

In [169]:
#translator = Translator(to_lang="en", from_lang="de", service="yandex")

data_dir = '../../../Data/Papyri Databases/PSI/'
img_dir  = '../../../Data/Papyri Databases/PSI/PSI_images/'

links_file = 'PSI_links.csv'
data_file  = 'PSI_metadata.csv'

df_links = pd.read_csv( data_dir + links_file )
nLinks = len(df_links)

df_links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3560 entries, 0 to 3559
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     3560 non-null   object
dtypes: object(1)
memory usage: 27.9+ KB


In [177]:
#doc_url = 'http://www.psi-online.it/documents'
base_url = 'http://www.psi-online.it/'

df_psi = pd.DataFrame(columns=['filename', 'page_url', 'img_url', 'title', 'inventory', 'typology',
                               'storage', 'origin', 'material', 'library_typology',
                               'content_r', 'content_v', 'dating', 'date', 'num_frags',
                               'dimensions',  'content', 'note', 'further_info',
                               'trismegistos', 'LDB_ext' ])

links = []
for i in range(2934,2936):
#for i in range(nLinks):
    #print(i)
    
    # check page url exists
    page_url = df_links.url.iloc[i]
    
    response = requests.get(page_url)
    
    if response.status_code != 200:
        print("Invalid: " + page_url)
        print()
        print()
        continue
    # end
    
    print("Valid:   " + page_url)
    
    # find table
    try:
        html = response.text
        
        soup = BeautifulSoup(html, 'html.parser')
        
        data = soup.body
        data = data.find('div', class_='container')
        data = data.find('div', id='content')
        data = data.find('section')
        data = data.find('table')
        table = data.find_all('td')[3:]
        
        #print(data.prettify())
        #print(data.text.strip())
    except:
        print("Failed to parse HTML")
        print()
        print()
        continue
    # end
    
    # find and download imgs
    print( "Images:" )
    tr_imgs = table[2].find_all('a')#.prettify()
    nImgs = len(tr_imgs)
    
    img_links = []
    img_files = []
    idx = []
    count2 = 0
    for j in range(nImgs):
        try:
            x = tr_imgs[j]['href']
            if x.split('/')[1] == 'images':
                img_url =  base_url + x[1:]
                img_file = re.sub( r'%20', '_', x.split('/')[-1])
                
                img_filepath = img_dir + img_file
                print( img_file, img_url )
                
                try:
                    #download_image(img_url, img_filepath, 'JPEG')
                    
                    idx.append(count2)
                    img_links.append( img_url )
                    img_files.append(img_file)
                    print("Download success")
                    count2 += 1
                except:
                    print("Download failed")
                # end
            # end
        except:
            1
        # end
    # end
    img_links = np.array(img_links)[idx]
    img_files = np.array(img_files)[idx]
    nImgs = len(img_links)
    
    if nImgs == 0:
        print("Failed to get images")
        print()
        print()
        continue
    # end
    
    # get title
    try:
        title = table[1].text.strip()
        if title == '':
            title = np.nan
        # end
    except:
        title = np.nan
    # end
    
    # get inventory
    try:
        inventory = table[4].text.strip()
        if title == '':
            title = np.nan
        # end
    except:
        inventory = np.nan
    # end
    
    # get typology
    try:
        typology = table[6].text.strip()
        if title == '':
            title = np.nan
        # end
    except:
        typology = np.nan
    # end
    
    # get storage
    try:
        storage = table[8].text.strip()
        if storage == '':
            storage = np.nan
        # end
    except:
        storage = np.nan
    # end
    
    # get origin
    try:
        origin = table[10].text.strip()
        if origin == '':
            origin = np.nan
        # end
    except:
        origin = np.nan
    # end
    
    # get material
    try:
        material = table[12].text.strip()
        if material == '':
            material = np.nan
        # end
    except:
        material = np.nan
    # end
    
    # get library_typology
    try:
        library_typology = table[14].text.strip()
        if library_typology == '':
            library_typology = np.nan
        # end
    except:
        library_typology = np.nan
    # end
    
    # get content_r
    try:
        content_r = table[16].text.strip()
        if content_r == '':
            content_r = np.nan
        # end
    except:
        content_r = np.nan
    # end
    
    # get content_v
    try:
        content_v = table[18].text.strip()
        if content_v == '':
            content_v = np.nan
        # end
    except:
        content_v = np.nan
    # end
    
    # get dating
    try:
        dating = table[20].text.strip()
        if dating == '':
            dating = np.nan
        # end
    except:
        dating = np.nan
    # end
    
    # get date
    try:
        date = table[22].text.strip()
        if date == '':
            date = np.nan
        # end
    except:
        date = np.nan
    # end
    
    # get num_frags
    try:
        num_frags = table[24].text.strip()
        if num_frags == '':
            num_frags = np.nan
        # end
    except:
        num_frags = np.nan
    # end
    
    # get dimensions
    try:
        dimensions = table[26].text.strip()
        if dimensions == '':
            dimensions = np.nan
        # end
    except:
        dimensions = np.nan
    # end
    
    # get content
    try:
        content = table[28].text.strip()
        if content == '':
            content = np.nan
        # end
    except:
        content = np.nan
    # end
    
    # get note
    try:
        note = table[30].text.strip()
        if note == '':
            note = np.nan
        # end
    except:
        note = np.nan
    # end
    
    # get further_info
    try:
        further_info = table[32].text.strip()
        if further_info == '':
            further_info = np.nan
        # end
    except:
        further_info = np.nan
    # end
    
    # get trismegistos
    try:
        trismegistos = table[34].text.strip()
        if trismegistos == '':
            trismegistos = np.nan
        # end
    except:
        trismegistos = np.nan
    # end
    
    # get LDB_ext
    try:
        LDB_ext = table[36].text.strip()
        if LDB_ext == '':
            LDB_ext = np.nan
        # end
    except:
        LDB_ext = np.nan
    # end
    
    # SAVE METADATA
    metadata = {}
    metadata['filename'] = img_files
    metadata['page_url'] = page_url
    metadata['img_url'] = img_links
    metadata['title'] = title
    metadata['inventory'] = inventory
    metadata['typology'] = typology
    metadata['storage'] = storage
    metadata['origin'] = origin
    metadata['material'] = material
    metadata['library_typology'] = library_typology
    metadata['content_r'] = content_r
    metadata['content_v'] = content_v
    metadata['dating'] = dating
    metadata['num_frags'] = num_frags
    metadata['dimensions'] = dimensions
    metadata['content'] = content
    metadata['note'] = note
    metadata['further_info'] = further_info
    metadata['trismegistos'] = trismegistos
    metadata['LDB_ext'] = LDB_ext
    
    df_psi = pd.concat([df_psi, pd.DataFrame(metadata)], ignore_index=True)
    
    print()
    print()
# end

Valid:   http://www.psi-online.it/documents/p-bastianini-24
Images:
PSI_inv_2702_r057.jpg http://www.psi-online.it/images/orig/PSI_inv_2702_r057.jpg
Download success
PSI_inv_2702_v058.jpg http://www.psi-online.it/images/orig/PSI_inv_2702_v058.jpg
Download success


Valid:   http://www.psi-online.it/documents/p-bastianini-16
Images:
PSI_inv_484_r063.jpg http://www.psi-online.it/images/orig/PSI_inv_484_r063.jpg
Download success
PSI_inv_484_v064.jpg http://www.psi-online.it/images/orig/PSI_inv_484_v064.jpg
Download success




In [176]:
metadata

{'filename': array(['PSI_I_79.jpg'], dtype='<U12'),
 'page_url': 'http://www.psi-online.it/documents/psi;1;79',
 'img_url': array(['http://www.psi-online.it/images/orig/PSI%20I%2079.jpg'],
       dtype='<U53'),
 'title': 'PSI I 79',
 'inventory': 'BML inv. 12114',
 'typology': 'Documentario',
 'storage': 'Firenze, Biblioteca Medicea Laurenziana',
 'origin': 'Oxyrhynchus',
 'material': 'Papiro',
 'library_typology': 'Testo documentario',
 'content_r': nan,
 'content_v': 'III 1 d.C.',
 'dating': '216/217 d.C.',
 'num_frags': 'cm 8,5 x 21,7',
 'dimensions': 'Contratto di vendita di due asine',
 'content': nan,
 'note': nan,
 'further_info': 'psi;1;79',
 'trismegistos': '20145',
 'LDB_ext': nan}

In [174]:
df_psi

,filename,page_url,img_url,title,inventory,typology,storage,origin,material,library_typology,content_r,content_v,dating,date,num_frags,dimensions,content,note,further_info,trismegistos,LDB_ext
0,PSI_I_79.jpg,http://www.psi-online.it/documents/psi;1;79,http://www.psi-online.it/images/orig/PSI%20I%2079.jpg,PSI I 79,BML inv. 12114,Documentario,"Firenze, Biblioteca Medicea Laurenziana",Oxyrhynchus,Papiro,Testo documentario,NaN,III 1 d.C.,216/217 d.C.,NaN,"cm 8,5 x 21,7",Contratto di vendita di due asine,NaN,NaN,psi;1;79,20145,NaN
1,PSI_I_80.jpg,http://www.psi-online.it/documents/psi;1;80,http://www.psi-online.it/images/orig/PSI%20I%2080.jpg,PSI I 80,BML inv. 12115,Documentario,"Firenze, Biblioteca Medicea Laurenziana",Oxyrhynchus,Papiro,Testo documentario,NaN,V d.C.,- -,NaN,"cm 53,3 x 35,2",Registro di tasse,"Il testo è su due colonne.Datazione originaria: VI d.C. La datazione al V d.C., qui riportata, è stata proposta in R.S. Bagnall, K.A. Worp, “The Chronological Systems of Byzantine Egypt” (Studia A...",NaN,psi;1;80,35295,NaN
2,PSI_I_81.jpg,http://www.psi-online.it/documents/psi;1;81,http://www.psi-online.it/images/orig/PSI%20I%2081.jpg,PSI I 81,BML inv. 12116,Documentario,"Firenze, Biblioteca Medicea Laurenziana",Oxyrhynchus,Papiro,Testo documentario,NaN,VI ex. d.C.,1 agosto 595 d.C.,NaN,"cm 6,4 x 15,5 (ca.)",Quietanza di affitto di officina,"Il papiro proviene dagli scavi condotti da G. Farina a Behnesa nel 1911.Datazione originaria: VI d.C. La datazione al 1.8.595 d.C., qui riportata, è stata proposta in BL VIII 392.Correzioni in BL ...",NaN,psi;1;81,20146,NaN
